# 01 Dataset Audit

Purpose: summarize the cleaned VIAME/DIVE dataset before training. This project uses bounding-box object detection only.

**Inputs**

- `annotations.viame.csv`, `annotations.dive.json`, `meta.json`, `image_manifest.csv`, and `class_counts.csv`
- `configs/experiments.yaml`

**Outputs**

- `outputs/dataset_audit/`
- `outputs/class_audit/`


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


## Run Dataset and Class Audits

The dataset audit summarizes image counts, annotation counts, class frequency, rare classes, and box-size bins. The class audit flags likely spelling or taxonomy issues for human review only; it does not rename classes.


In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {DATASET_ROOT}. Set FISHDETECT_DATASET_ROOT and rerun.")

!python scripts/audit_dataset.py --config configs/experiments.yaml --out outputs/dataset_audit
!python scripts/audit_classes.py --dataset "$FISHDETECT_DATASET_ROOT" --out outputs/class_audit


## Audit Summary


In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

summary_path = Path("outputs/dataset_audit/dataset_audit_summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(Markdown(
        f"""
**Images:** {summary.get('image_count', 'n/a')}  
**Annotations:** {summary.get('annotation_count', 'n/a')}  
**Classes:** {summary.get('class_count', 'n/a')}  
**Multi-fish images:** {summary.get('multi_fish_image_count', 'n/a')}  
**Multi-class images:** {summary.get('multi_class_image_count', 'n/a')}  
"""
    ))
else:
    print("Dataset audit summary was not created. Review the audit command output above.")

for label, path in [
    ("Class frequency", Path("outputs/dataset_audit/class_frequency.csv")),
    ("Rare classes", Path("outputs/dataset_audit/rare_classes.csv")),
    ("Bounding-box size bins", Path("outputs/dataset_audit/bbox_size_bins.csv")),
    ("Class-name review flags", Path("outputs/class_audit/class_audit.csv")),
]:
    print(f"
{label}")
    if path.exists():
        df = pd.read_csv(path)
        if label == "Class-name review flags" and "issue_type" in df.columns:
            df = df[df["issue_type"].fillna("") != ""]
        display(df.head(25))
    else:
        print(f"Missing: {path}")


## Interpretation Notes

Use this audit to understand imbalance, rare classes, crowded images, and likely taxonomy review items before training. Class labels are not changed automatically in this workflow.
